# 05 - Model Training

Train and compare multiple demand forecasting models.

## Objectives:
- Train all five model types (Baseline, Random Forest, XGBoost, LightGBM, Prophet)
- Compare performance using MAE, RMSE, MAPE
- Select and save the best model
- Save encoders and scaler for production

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.forecasting import DemandForecaster
from src.preprocessing import DataPreprocessor
from src.utils import load_dataframe, save_model

%matplotlib inline

## 1. Load Engineered Features

In [ ]:
df = load_dataframe('../data/processed/engineered_features.csv')
df['date'] = pd.to_datetime(df['date'])

print(f"Data loaded: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nTarget variable (sales) statistics:")
print(df['sales'].describe())

## 2. Train All Models & Compare

Using a 10% sample for speed. Remove sampling for production training.

In [ ]:
print("=" * 60)
print("TRAINING ALL MODELS — COMPARISON")
print("=" * 60)

forecaster = DemandForecaster()
comparison_df, best_model = forecaster.train_all_models(
    df, target_col='sales', test_size=0.2, sample_frac=0.1
)

print("\n" + "=" * 60)
print("MODEL COMPARISON RESULTS")
print("=" * 60)
display(comparison_df.round(4))

print(f"\n{'='*60}")
print(f"BEST MODEL: {best_model}")
print("=" * 60)

## 3. Visualize Comparison

In [ ]:
comparison_plot = comparison_df.dropna(subset=['MAE']).copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# MAE
axes[0].bar(comparison_plot['model'], comparison_plot['MAE'], color=['#2C5282', '#E53E3E', '#38A169', '#D69E2E', '#805AD5'])
axes[0].set_title('MAE (lower is better)')
axes[0].set_ylabel('MAE')
axes[0].tick_params(axis='x', rotation=45)

# RMSE
axes[1].bar(comparison_plot['model'], comparison_plot['RMSE'], color=['#2C5282', '#E53E3E', '#38A169', '#D69E2E', '#805AD5'])
axes[1].set_title('RMSE (lower is better)')
axes[1].set_ylabel('RMSE')
axes[1].tick_params(axis='x', rotation=45)

# MAPE
axes[2].bar(comparison_plot['model'], comparison_plot['MAPE'], color=['#2C5282', '#E53E3E', '#38A169', '#D69E2E', '#805AD5'])
axes[2].set_title('MAPE % (lower is better)')
axes[2].set_ylabel('MAPE %')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Training time chart
plt.figure(figsize=(10, 4))
plt.bar(comparison_plot['model'], comparison_plot['training_time_sec'], color=['#2C5282', '#E53E3E', '#38A169', '#D69E2E', '#805AD5'])
plt.title('Training Time (seconds)')
plt.ylabel('Seconds')
plt.tick_params(axis='x', rotation=45)
plt.show()

## 4. Train Best Model on Full Data

In [ ]:
print(f"Training {best_model} on full dataset...")

best_forecaster = DemandForecaster(model_type=best_model)
best_model_obj, test_scores = best_forecaster.train_model(df, target_col='sales')

print("\n" + "=" * 60)
print("BEST MODEL — FINAL TEST PERFORMANCE")
print("=" * 60)
for metric, value in test_scores.items():
    print(f"  {metric}: {value:.4f}")

## 5. Save Best Model + Comparison

In [ ]:
best_forecaster.save_model('../models/best_model.joblib')

# Also save the comparison for reference
comparison_df.to_csv('../reports/exports/model_comparison.csv', index=False)

print(f"\nModel saved to: models/best_model.joblib")
print(f"Model type: {best_model}")
print(f"Comparison saved to: reports/exports/model_comparison.csv")
print(f"\n{'='*60}")
print("MODEL PERFORMANCE SUMMARY")
print("=" * 60)
for metric, value in test_scores.items():
    print(f"  {metric:15s}: {value:10.4f}")
print("=" * 60)

## 6. Save Encoders & Scaler

In [ ]:
from src.preprocessing import DataPreprocessor
from src.data_loader import DataLoader

# Re-fit and save label encoders for production
loader = DataLoader(data_dir='../data/raw')
prices_df = loader.load_prices()
calendar_df = loader.load_calendar()

preprocessor = DataPreprocessor()
calendar_clean = preprocessor.clean_calendar(calendar_df)
prices_clean = preprocessor.clean_prices(prices_df)

# Encode common categorical columns
cat_cols = ['item_id', 'store_id', 'cat_id', 'dept_id', 'state_id']
for col in cat_cols:
    if col in df.columns:
        preprocessor.encode_categorical(df, [col])

preprocessor.save_encoders('../models/encoders.joblib')
print("Encoders saved to models/encoders.joblib")

## Summary

Model training completed:
- Baseline model trained
- Random Forest trained
- XGBoost trained
- LightGBM trained
- Prophet trained
- Models compared on MAE, RMSE, MAPE
- Best model selected and saved
- Encoders saved for production

Next: Model Evaluation & SHAP Analysis